# 🏦 Lab 1 — Bond Math on Real Market Data

**MIT 18.642 — Topics in Mathematics with Applications in Finance**

---

This notebook takes every concept from Lecture 1 (compounding, discounting, yield curves,
bond pricing, duration, convexity) and applies them to **real US Treasury data**.

| Section | What You'll Do |
|---------|----------------|
| 1 | Fetch (or load) the real US Treasury yield curve |
| 2 | Plot the yield curve and interpret its shape |
| 3 | Price a real 10-year Treasury note |
| 4 | Compute Duration & Convexity and stress-test the bond |
| 5 | Compare historical yield-curve shapes and economic signals |

> **Prerequisites:** `numpy`, `pandas`, `matplotlib`, `scipy`. Optional: `fredapi`.

---

In [ ]:
# ── Core Imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

# Professional dark plot style
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'font.family': 'sans-serif',
    'axes.edgecolor': '#444444',
    'axes.linewidth': 0.8,
    'grid.alpha': 0.25,
})

# Accent colour palette
CYAN   = '#00E5FF'
PINK   = '#FF4081'
AMBER  = '#FFD740'
GREEN  = '#69F0AE'
PURPLE = '#B388FF'
WHITE  = '#E0E0E0'

print('✅ All imports loaded.')

---
## Section 1 — Fetch Real Treasury Yield Curve Data

We pull US Treasury **Constant Maturity Rates** from the Federal Reserve's
[FRED database](https://fred.stlouisfed.org/). These are the benchmark
risk-free rates for the entire US fixed-income market.

| FRED Series | Maturity |
|-------------|----------|
| DGS1MO | 1 Month |
| DGS3MO | 3 Months |
| DGS6MO | 6 Months |
| DGS1   | 1 Year |
| DGS2   | 2 Years |
| DGS3   | 3 Years |
| DGS5   | 5 Years |
| DGS7   | 7 Years |
| DGS10  | 10 Years |
| DGS20  | 20 Years |
| DGS30  | 30 Years |

> **Tip:** To use `fredapi`, get a free API key at https://fred.stlouisfed.org/docs/api/api_key.html
> and set `FRED_API_KEY` below. If you don't have one, the **hardcoded fallback** runs fine.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
FRED_API_KEY = None  # ← paste your key as a string, e.g. 'abc123...'

# FRED series IDs mapped to maturity in years
SERIES = {
    'DGS1MO': 1/12,
    'DGS3MO': 3/12,
    'DGS6MO': 6/12,
    'DGS1':   1,
    'DGS2':   2,
    'DGS3':   3,
    'DGS5':   5,
    'DGS7':   7,
    'DGS10':  10,
    'DGS20':  20,
    'DGS30':  30,
}

# ── Hardcoded fallback (approximate mid-2024 yields, %) ──────────────────────
FALLBACK_YIELDS = {
    'DGS1MO': 5.50,
    'DGS3MO': 5.40,
    'DGS6MO': 5.30,
    'DGS1':   5.10,
    'DGS2':   4.70,
    'DGS3':   4.40,
    'DGS5':   4.20,
    'DGS7':   4.20,
    'DGS10':  4.30,
    'DGS20':  4.50,
    'DGS30':  4.40,
}

In [ ]:
def fetch_yield_curve(api_key=None):
    """
    Try to pull the latest Treasury yields from FRED.
    Falls back to hardcoded mid-2024 data if fredapi is unavailable or key is None.
    
    Returns
    -------
    maturities : np.ndarray  — tenor in years
    yields_pct : np.ndarray  — annualised yield in percent
    source     : str         — 'FRED (live)' or 'Hardcoded fallback (mid-2024)'
    """
    if api_key:
        try:
            from fredapi import Fred
            fred = Fred(api_key=api_key)
            data = {}
            for sid in SERIES:
                s = fred.get_series(sid)
                # Take the most recent non-NaN observation
                val = s.dropna().iloc[-1]
                data[sid] = val
            maturities = np.array([SERIES[k] for k in data])
            yields_pct = np.array(list(data.values()))
            return maturities, yields_pct, 'FRED (live)'
        except Exception as e:
            print(f'⚠️  FRED fetch failed ({e}). Using fallback data.')

    # Fallback path
    maturities = np.array([SERIES[k] for k in FALLBACK_YIELDS])
    yields_pct = np.array(list(FALLBACK_YIELDS.values()))
    return maturities, yields_pct, 'Hardcoded fallback (mid-2024)'


maturities, yields_pct, data_source = fetch_yield_curve(FRED_API_KEY)
yields_dec = yields_pct / 100  # decimal form for calculations

# Build a tidy DataFrame for display
labels = ['1M','3M','6M','1Y','2Y','3Y','5Y','7Y','10Y','20Y','30Y']
df_curve = pd.DataFrame({
    'Tenor':        labels,
    'Maturity (yr)': maturities,
    'Yield (%)':     yields_pct,
})

print(f'📡 Data source: {data_source}\n')
df_curve

---
## Section 2 — Plot the Yield Curve

The **yield curve** is the single most important chart in fixed-income markets.
Its shape tells you what the market expects about future interest rates and the
economy:

| Shape | What It Looks Like | Economic Signal |
|-------|--------------------|-----------------|
| **Normal** | Slopes upward | Growth expected; investors demand extra yield for locking up money longer |
| **Flat** | Nearly horizontal | Uncertainty; transition between regimes |
| **Inverted** | Slopes downward | Recession signal; short rates above long rates (tight monetary policy) |
| **Humped** | Peaks in the middle | Mixed signals; often precedes inversion |

### What to look for below
- The mid-2024 curve is **inverted** — short-term yields (1M–1Y) are *higher*
  than long-term yields (5Y–30Y). This is a classic signal of tight Fed policy
  and market expectation of future rate cuts.

In [ ]:
# ── Smooth interpolation for a pretty curve ────────────────────────────────────
cs = CubicSpline(maturities, yields_pct)
t_smooth = np.linspace(maturities.min(), maturities.max(), 300)
y_smooth = cs(t_smooth)

fig, ax = plt.subplots(figsize=(13, 6.5))

# Filled area under the curve
ax.fill_between(t_smooth, y_smooth, alpha=0.10, color=CYAN)

# Interpolated smooth line
ax.plot(t_smooth, y_smooth, color=CYAN, linewidth=2.5, label='Interpolated curve')

# Actual data points
ax.scatter(maturities, yields_pct, color=PINK, s=80, zorder=5, edgecolors='white',
           linewidths=0.5, label='Observed yields')

# Label each data point
for lbl, m, y in zip(labels, maturities, yields_pct):
    ax.annotate(f'{lbl}\n{y:.2f}%', (m, y),
                textcoords='offset points', xytext=(0, 14),
                ha='center', fontsize=8.5, color=WHITE,
                arrowprops=dict(arrowstyle='-', color='#555555', lw=0.5))

# Shape annotation
peak_y = yields_pct.max()
trough_y = yields_pct[maturities >= 2].min()
if peak_y - trough_y > 0.5 and np.argmax(yields_pct) < len(yields_pct) // 2:
    shape_label = '⚠️ INVERTED — short rates > long rates'
    shape_color = PINK
elif peak_y - trough_y < 0.3:
    shape_label = '➖ FLAT curve'
    shape_color = AMBER
else:
    shape_label = '📈 NORMAL — upward sloping'
    shape_color = GREEN

ax.text(0.98, 0.05, shape_label, transform=ax.transAxes, fontsize=13,
        ha='right', va='bottom', color=shape_color,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#1a1a2e', edgecolor=shape_color, alpha=0.9))

ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Yield (%)')
ax.set_title(f'US Treasury Yield Curve  ·  Source: {data_source}', fontweight='bold')
ax.legend(loc='upper left', framealpha=0.6)
ax.grid(True, linestyle='--')
fig.tight_layout()
plt.show()

---
## Section 3 — Price a Real Treasury Bond

Let's price an actual US Treasury note:

| Feature | Value |
|---------|-------|
| Coupon Rate | 4.25% (semi-annual) |
| Face Value | \$1,000 |
| Maturity | 10 years |
| Issue | ~May 2024 |

### Method
We **discount each cash flow** using the interpolated yield curve
(bootstrapped zero rates). This is more realistic than using a single flat
yield — each cash flow gets discounted at the rate appropriate for *its* tenor.

$$P = \sum_{i=1}^{N} \frac{C/2}{\bigl(1 + y(t_i)/2\bigr)^{2t_i}} + \frac{F}{\bigl(1 + y(T)/2\bigr)^{2T}}$$

where $y(t_i)$ is the interpolated yield at tenor $t_i$.

In [ ]:
# ── Bond specification ────────────────────────────────────────────────────────
COUPON_RATE  = 0.0425     # annual coupon rate (4.25%)
FACE         = 1000.0     # par / face value ($)
MATURITY_YRS = 10         # years to maturity
FREQ         = 2          # semi-annual coupons

# Semi-annual coupon payment
coupon_payment = FACE * COUPON_RATE / FREQ  # $21.25 every 6 months

# Cash-flow schedule: every 0.5 years up to 10 years
cf_times = np.arange(1/FREQ, MATURITY_YRS + 1/FREQ, 1/FREQ)  # [0.5, 1.0, ... , 10.0]
n_coupons = len(cf_times)

print(f'Bond: {COUPON_RATE*100:.2f}% coupon, {MATURITY_YRS}Y maturity, semi-annual')
print(f'Coupon payment: ${coupon_payment:.2f} every 6 months')
print(f'Number of coupon periods: {n_coupons}')
print(f'Cash-flow times (years): {cf_times}')

In [ ]:
def price_bond_from_curve(coupon_pmt, face, cf_times, maturities, yields_dec, freq=2):
    """
    Price a bond by discounting each cash flow at the
    interpolated yield-curve rate for that tenor.
    
    Parameters
    ----------
    coupon_pmt : float  — dollar coupon per period
    face       : float  — face / par value
    cf_times   : array  — times (in years) of each cash flow
    maturities : array  — observed tenor grid points (years)
    yields_dec : array  — observed yields in decimal (e.g., 0.043)
    freq       : int    — compounding frequency per year
    
    Returns
    -------
    price      : float
    pv_flows   : array  — present value of each cash flow
    y_interp   : array  — yield used to discount each cash flow
    """
    # Cubic spline interpolation of the yield curve (in decimal)
    cs = CubicSpline(maturities, yields_dec)
    
    # Clamp interpolation to the observed range
    y_interp = cs(np.clip(cf_times, maturities.min(), maturities.max()))
    
    # Discount each cash flow
    cash_flows = np.full_like(cf_times, coupon_pmt)
    cash_flows[-1] += face  # add principal repayment at maturity
    
    # Discount factor: (1 + y/freq)^(freq * t)
    disc_factors = (1 + y_interp / freq) ** (freq * cf_times)
    pv_flows = cash_flows / disc_factors
    
    price = pv_flows.sum()
    return price, pv_flows, y_interp


calc_price, pv_flows, y_at_cf = price_bond_from_curve(
    coupon_payment, FACE, cf_times, maturities, yields_dec, FREQ
)

# Approximate market price for comparison (mid-2024 for a 4.25% 10Y note
# trading near par when yield ≈ coupon)
MARKET_PRICE = 995.50  # approximate observed clean price

print(f'\n💰 Calculated Bond Price:  ${calc_price:,.2f}')
print(f'📊 Approx. Market Price:   ${MARKET_PRICE:,.2f}')
print(f'   Difference:             ${calc_price - MARKET_PRICE:+,.2f}  '
      f'({(calc_price - MARKET_PRICE)/MARKET_PRICE*100:+.3f}%)')

In [ ]:
# ── Cash-flow waterfall chart ─────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: Nominal vs PV of each cash flow
nominal_cfs = np.full_like(cf_times, coupon_payment)
nominal_cfs[-1] += FACE

bar_w = 0.18
ax1.bar(cf_times - bar_w/2, nominal_cfs, width=bar_w, color=AMBER, alpha=0.7, label='Nominal CF')
ax1.bar(cf_times + bar_w/2, pv_flows,    width=bar_w, color=CYAN,  alpha=0.8, label='PV of CF')
ax1.set_xlabel('Time (years)')
ax1.set_ylabel('Cash Flow ($)')
ax1.set_title('Nominal vs Present-Value Cash Flows', fontweight='bold')
ax1.legend(framealpha=0.6)
ax1.grid(True, linestyle='--')

# Right: Discount rate used at each tenor
ax2.plot(cf_times, y_at_cf * 100, 'o-', color=GREEN, markersize=5)
ax2.set_xlabel('Cash-Flow Tenor (years)')
ax2.set_ylabel('Interpolated Yield (%)')
ax2.set_title('Yield Used to Discount Each Cash Flow', fontweight='bold')
ax2.grid(True, linestyle='--')

fig.suptitle(f'10Y Treasury Note · Calculated Price = ${calc_price:,.2f}',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

---
## Section 4 — Duration & Convexity of a Real Bond

### Recap
- **Macaulay Duration** — the weighted-average time to receive cash flows
  (weights = PV of each flow / price). *Units: years.*
- **Modified Duration** — how much the price changes (%) per 1% change in yield.
  $$D_{\text{mod}} = \frac{D_{\text{mac}}}{1 + y/m}$$
- **Convexity** — the curvature correction (second-order term).

### Why it matters
If yields move by $\Delta y$:

$$\frac{\Delta P}{P} \approx -D_{\text{mod}} \cdot \Delta y + \tfrac{1}{2}\,C \cdot (\Delta y)^2$$

Duration alone *under-estimates* price gains when yields fall and
*over-estimates* losses when yields rise. Convexity fixes that.

In [ ]:
def bond_analytics(coupon_pmt, face, cf_times, ytm, freq=2):
    """
    Compute price, Macaulay Duration, Modified Duration, and Convexity
    using a flat yield-to-maturity (for the analytical formulas).
    
    Parameters
    ----------
    coupon_pmt : float  — dollar coupon per period
    face       : float  — face value
    cf_times   : array  — cash-flow times in years
    ytm        : float  — yield to maturity (decimal, annual)
    freq       : int    — coupons per year
    
    Returns
    -------
    dict with keys: price, mac_dur, mod_dur, convexity
    """
    # Build cash-flow vector
    cfs = np.full_like(cf_times, coupon_pmt)
    cfs[-1] += face
    
    # Discount factors at the flat YTM
    disc = (1 + ytm / freq) ** (freq * cf_times)
    pv = cfs / disc
    price = pv.sum()
    
    # Macaulay Duration = sum(t_i * PV_i) / Price
    mac_dur = (cf_times * pv).sum() / price
    
    # Modified Duration = Mac / (1 + y/m)
    mod_dur = mac_dur / (1 + ytm / freq)
    
    # Dollar Convexity  (per-period formula, then adjust)
    # C = (1/P) * sum[ t_i*(t_i + 1/freq) * PV_i ] / (1 + y/freq)^2
    convexity = (1 / price) * (cf_times * (cf_times + 1/freq) * pv).sum() / (1 + ytm/freq)**2
    
    return {
        'price':     price,
        'mac_dur':   mac_dur,
        'mod_dur':   mod_dur,
        'convexity': convexity,
    }


# Use the 10Y yield as the flat YTM for this bond
ytm_10y = yields_dec[labels.index('10Y')]
analytics = bond_analytics(coupon_payment, FACE, cf_times, ytm_10y, FREQ)

print(f'Using YTM = {ytm_10y*100:.2f}%\n')
print(f'  Bond Price (flat YTM):    ${analytics["price"]:,.2f}')
print(f'  Macaulay Duration:         {analytics["mac_dur"]:.4f} years')
print(f'  Modified Duration:         {analytics["mod_dur"]:.4f}')
print(f'  Convexity:                 {analytics["convexity"]:.4f}')

In [ ]:
# ── Yield Shock Analysis ──────────────────────────────────────────────────────
# Simulate yield changes from -150 bps to +150 bps
bps_range = np.linspace(-150, 150, 301)  # in basis points
dy = bps_range / 10000  # convert to decimal

P0   = analytics['price']
Dmod = analytics['mod_dur']
C    = analytics['convexity']

# Actual re-priced values
actual_prices = np.array([
    bond_analytics(coupon_payment, FACE, cf_times, ytm_10y + d, FREQ)['price']
    for d in dy
])
actual_pct_change = (actual_prices - P0) / P0 * 100

# Duration-only estimate:  ΔP/P ≈ -Dmod * Δy
dur_estimate = -Dmod * dy * 100

# Duration + Convexity estimate:  ΔP/P ≈ -Dmod*Δy + 0.5*C*(Δy)²
dur_conv_estimate = (-Dmod * dy + 0.5 * C * dy**2) * 100

# ── Table for specific shocks ─────────────────────────────────────────────────
shock_bps = [-100, -50, 50, 100]
rows = []
for s in shock_bps:
    d = s / 10000
    new_p = bond_analytics(coupon_payment, FACE, cf_times, ytm_10y + d, FREQ)['price']
    actual_chg = (new_p - P0) / P0 * 100
    dur_chg    = -Dmod * d * 100
    dc_chg     = (-Dmod * d + 0.5 * C * d**2) * 100
    rows.append({
        'Shock (bps)':       f'{s:+d}',
        'New Yield (%)':     f'{(ytm_10y + d)*100:.2f}',
        'New Price ($)':     f'{new_p:,.2f}',
        'Actual ΔP/P (%)':   f'{actual_chg:+.3f}',
        'Duration Est (%)':  f'{dur_chg:+.3f}',
        'Dur+Conv Est (%)':  f'{dc_chg:+.3f}',
    })

df_shocks = pd.DataFrame(rows)
print('📉📈 Yield Shock Comparison\n')
df_shocks

In [ ]:
# ── Visualise the approximation quality ───────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.5))

# Left plot: Price change (%) vs yield shock
ax1.plot(bps_range, actual_pct_change,    color=CYAN,   lw=2.5, label='Actual (re-priced)')
ax1.plot(bps_range, dur_estimate,         color=PINK,   lw=2,   ls='--', label='Duration only')
ax1.plot(bps_range, dur_conv_estimate,    color=GREEN,  lw=2,   ls='-.', label='Duration + Convexity')

# Mark the specific shocks
for s in shock_bps:
    idx = list(bps_range).index(s) if s in bps_range else np.argmin(np.abs(bps_range - s))
    ax1.axvline(s, color=AMBER, alpha=0.3, lw=0.8, ls=':')

ax1.axhline(0, color='white', lw=0.5, alpha=0.4)
ax1.axvline(0, color='white', lw=0.5, alpha=0.4)
ax1.set_xlabel('Yield Shock (bps)')
ax1.set_ylabel('Price Change (%)')
ax1.set_title('Price Sensitivity: Actual vs Approximations', fontweight='bold')
ax1.legend(framealpha=0.6)
ax1.grid(True, ls='--')

# Right plot: Approximation error
err_dur  = dur_estimate - actual_pct_change
err_conv = dur_conv_estimate - actual_pct_change

ax2.plot(bps_range, err_dur,  color=PINK,  lw=2, ls='--', label='Duration error')
ax2.plot(bps_range, err_conv, color=GREEN, lw=2, ls='-',  label='Dur+Conv error')
ax2.axhline(0, color='white', lw=0.5, alpha=0.4)
ax2.set_xlabel('Yield Shock (bps)')
ax2.set_ylabel('Approximation Error (pp)')
ax2.set_title('How Much Does Each Approximation Miss?', fontweight='bold')
ax2.legend(framealpha=0.6)
ax2.grid(True, ls='--')

fig.tight_layout()
plt.show()

print('\n🔑 Key Insight: Duration alone is a straight line (first-order Taylor approx).')
print('   Adding convexity captures the curvature and is far more accurate for large shocks.')

---
## Section 5 — Historical Yield Curve Analysis

To understand *why* the yield curve shape matters, let's compare three
snapshots from very different economic environments:

| Date | Context | Curve Shape |
|------|---------|-------------|
| **Nov 2018** | Mid-cycle expansion, Fed hiking | Normal (gently upward) |
| **Jul 2023** | Post-pandemic inflation, Fed at peak rates | Deeply inverted |
| **Mid-2024** | Fed holding, rate cuts priced in | Inverted but flattening |

### What does inversion mean?
When the **2Y yield > 10Y yield**, the market is effectively saying:
- Short-term rates are high *right now* (Fed tightening).
- But the market expects rates to be *lower* in the future (economic slowdown → Fed cuts).

Historically, a sustained 2s10s inversion has preceded every US recession
since the 1960s (with varying lead times of 6–24 months).

In [ ]:
# ── Historical yield curve snapshots (approximate, from FRED) ─────────────────
# Maturities in years (same grid for all)
hist_maturities = np.array([1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30])

# November 2018: Normal, gently upward-sloping curve
yields_2018 = np.array([2.27, 2.38, 2.56, 2.68, 2.83, 2.84, 2.88, 2.96, 3.07, 3.21, 3.30])

# July 2023: Deeply inverted — Fed at peak rates
yields_2023 = np.array([5.54, 5.52, 5.46, 5.40, 4.87, 4.49, 4.18, 4.09, 3.96, 4.20, 4.02])

# Mid-2024: Still inverted but less extreme (our current dataset)
yields_2024 = yields_pct

# ── Build comparison DataFrames ───────────────────────────────────────────────
df_hist = pd.DataFrame({
    'Tenor': labels,
    'Nov 2018 (%)': yields_2018,
    'Jul 2023 (%)': yields_2023,
    'Mid-2024 (%)': yields_2024,
})
print('Historical Yield Curve Snapshots\n')
df_hist

In [ ]:
# ── Historical comparison plot ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

# Smooth interpolations for each era
t_fine = np.linspace(hist_maturities.min(), hist_maturities.max(), 300)

curves = [
    ('Nov 2018 — Normal',    yields_2018, CYAN,   '-',  '📈 Growth outlook positive'),
    ('Jul 2023 — Inverted',  yields_2023, PINK,   '--', '⚠️  Deep inversion → recession signal'),
    ('Mid-2024 — Inverted (flattening)', yields_2024, AMBER, '-.', '🔄 Market pricing in rate cuts'),
]

for name, ydata, color, ls, _ in curves:
    cs_h = CubicSpline(hist_maturities, ydata)
    ax.plot(t_fine, cs_h(t_fine), color=color, lw=2.5, ls=ls, label=name)
    ax.scatter(hist_maturities, ydata, color=color, s=40, zorder=5, edgecolors='white', lw=0.4)

# Annotation box with economic context
annotation_text = '\n'.join([f'{name}: {note}' for name, _, _, _, note in curves])
ax.text(0.02, 0.02, annotation_text, transform=ax.transAxes,
        fontsize=10, va='bottom', ha='left', color=WHITE,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#1a1a2e', edgecolor='#444444', alpha=0.9))

# 2s10s spread indicator
for name, ydata, color, _, _ in curves:
    spread_2s10s = ydata[labels.index('10Y')] - ydata[labels.index('2Y')]
    y_pos = ydata[labels.index('10Y')]
    ax.annotate(f'2s10s: {spread_2s10s:+.0f}bp',
                xy=(10, y_pos), fontsize=8.5, color=color,
                textcoords='offset points', xytext=(15, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=0.8))

ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Yield (%)')
ax.set_title('US Treasury Yield Curve — Historical Comparison', fontsize=16, fontweight='bold')
ax.legend(loc='upper right', framealpha=0.7, fontsize=11)
ax.grid(True, ls='--')
fig.tight_layout()
plt.show()

print('\n📊 2s10s Spread Summary:')
for name, ydata, _, _, _ in curves:
    s = ydata[labels.index('10Y')] - ydata[labels.index('2Y')]
    print(f'   {name}: {s:+.0f} bps  {"(INVERTED)" if s < 0 else "(NORMAL)"}')

---
## 🎓 Key Takeaways

1. **The yield curve is not flat.** Pricing a bond with a single rate
   is a simplification. Using the full term structure (one rate per tenor)
   gives a more accurate price.

2. **Duration is a first-order approximation.** It works well for small
   yield changes (≤ 25 bps) but misses the curvature for larger moves.
   Adding convexity makes the estimate dramatically better.

3. **Yield curve shape = economic crystal ball.** An inverted curve has
   historically been one of the most reliable recession predictors.
   The 2022–2024 inversion was the deepest since the early 1980s.

4. **Convexity is your friend.** Bonds have positive convexity — they
   gain more when yields fall than they lose when yields rise by the same
   amount. This asymmetry is valuable to bondholders.

---

**Next up:** Lecture 2 — Linear Algebra (matrix operations, eigenvalues,
and their applications in portfolio theory and PCA of yield curves).